# 04 — Phase 3b: Retraining on Windowed Features

Tests whether Phase 2's windowed velocity features (per-account transaction count, total
and mean amount, and time since the last transaction, over 30-second sliding windows)
help detect fraud, framed two ways: labeling whole windows, and labeling individual
transactions with their window's features attached.

**Assumes:** `data/processed/windowed_features_batch.jsonl`, built by
`python -m fraud_pipeline.features.batch --duration-minutes 10`. That command runs the
same windowing and aggregation code as the streaming pipeline over the CSV, on the
producer's 10-minute replay clock, so its windows match what a live replay of the same
length produces. See `docs/commands.md`.

**Why batch, not a live capture:** the first version of this notebook trained on windows
captured from a live streaming run. Those windows held only about 0.6% of the
transactions the labels counted: a median of 1 transaction per window, against 37 in
ground truth for the same account and time range. Labels came from a time-range join
against all of ground truth, so most of them described transactions the features never
saw. The results it reported (a 20% positive rate, 0.79 ROC-AUC) are superseded by this
version. `docs/phase3-decisions.md` has the details.

In [1]:
import sys

sys.path.insert(0, "..")

import pandas as pd

from fraud_pipeline.ingestion.loader import RAW_DATA_PATH
from fraud_pipeline.ingestion.producer import load_and_prepare
from fraud_pipeline.models.evaluation import evaluate
from fraud_pipeline.models.windowed_dataset import (
    FEATURE_COLUMNS,
    build_labeled_windowed_dataset,
    check_window_coverage,
    load_jsonl,
    scoring_windows,
    window_fraud_stats,
)
from fraud_pipeline.models.xgboost_baseline import fit_xgboost
from fraud_pipeline.models.xgboost_baseline import score as score_xgboost

windows = load_jsonl("../data/processed/windowed_features_batch.jsonl")

# Same seed and account count as the producer's defaults, so transaction_id ->
# account_id matches the windows (and any live replay with the same settings).
transactions = load_and_prepare(str(RAW_DATA_PATH), None, num_accounts=2000, seed=42)
labels = transactions.rename(columns={"Class": "class"})[["transaction_id", "class"]]

print(f"windows: {len(windows):,}")
print(f"transactions: {len(transactions):,}, fraud: {labels['class'].sum()}")

windows: 69,639
transactions: 284,807, fraud: 492


## Every transaction is in exactly three windows

30-second windows every 10 seconds means each transaction belongs to exactly 3 windows.
`check_window_coverage` raises if any transaction is missing, appears a different number
of times, or if a window's `transaction_count` disagrees with the IDs it lists. The first
capture would have failed this check badly.

In [2]:
check_window_coverage(windows, transactions["transaction_id"], 30, 10)

print("coverage check passed")
print(f"sum of transaction_count: {windows['transaction_count'].sum():,} "
      f"= 3 x {len(transactions):,}")
print(f"transactions per window: median {windows['transaction_count'].median():.0f}, "
      f"mean {windows['transaction_count'].mean():.1f}, max {windows['transaction_count'].max():,}")

coverage check passed
sum of transaction_count: 854,421 = 3 x 284,807
transactions per window: median 2, mean 12.3, max 5,206


## Window-level labels: how much fraud is in a "fraud" window?

The original rule labels a window fraud if **any** of its transactions is fraud. With the
labels now joined by transaction ID, it's possible to see what that rule actually
selects.

In [3]:
stats = window_fraud_stats(windows, labels)
fraud_windows = stats[stats["fraud_count"] >= 1]

rule_rates = pd.DataFrame(
    {
        "windows labeled fraud": [
            int((stats["fraud_count"] >= 1).sum()),
            *[int((stats["fraud_fraction"] >= f).sum()) for f in (0.1, 0.25, 0.5, 1.0)],
        ],
    },
    index=["any fraud", ">= 10% fraud", ">= 25% fraud", ">= 50% fraud", "100% fraud"],
)
rule_rates["share of windows"] = (rule_rates["windows labeled fraud"] / len(windows)).map(
    "{:.3%}".format
)
print(f"raw transaction fraud rate: {labels['class'].mean():.3%}")
rule_rates

raw transaction fraud rate: 0.173%


,windows labeled fraud,share of windows
any fraud,971,1.394%
>= 10% fraud,293,0.421%
>= 25% fraud,194,0.279%
>= 50% fraud,132,0.190%
100% fraud,67,0.096%


In [4]:
quantiles = [0.1, 0.25, 0.5, 0.75, 0.9]
pd.DataFrame(
    {
        "fraud share of the window": fraud_windows["fraud_fraction"].quantile(quantiles).round(3),
        "transactions in the window": windows.loc[fraud_windows.index, "transaction_count"]
        .quantile(quantiles)
        .round(0),
    }
)

,fraud share of the window,transactions in the window
0.10,0.002,2.0
0.25,0.006,7.0
0.50,0.024,44.0
0.75,0.143,231.0
0.90,0.500,1112.0


Under the "any fraud" rule, 1.4% of windows are positive: about 8× the raw fraud rate,
not the 116× the first capture suggested. More important is what those positive windows
contain. In the median fraud window, fraud is 2.4% of the transactions and the window
holds 44 of them. The window's aggregate features mostly describe legitimate activity,
and stricter rules leave too few positives to train on (132 windows at a 50%
threshold).

## Window-level model: the label mostly measures how busy the window is

Temporal 70/30 split on `window_start`, the same discipline as Phase 3a.

In [5]:
from sklearn.metrics import average_precision_score, roc_auc_score

windowed = build_labeled_windowed_dataset(windows, labels, rule="any")
windowed["window_start"] = pd.to_datetime(windowed["window_start"], format="ISO8601")
windowed = windowed.sort_values("window_start").reset_index(drop=True)
cutoff = windowed["window_start"].quantile(0.70)
w_train = windowed[windowed["window_start"] <= cutoff]
w_test = windowed[windowed["window_start"] > cutoff]
print(f"train: {len(w_train):,} windows, {w_train['label'].mean():.2%} positive")
print(f"test:  {len(w_test):,} windows, {w_test['label'].mean():.2%} positive")

xgb_window = fit_xgboost(w_train[FEATURE_COLUMNS].values, w_train["label"].values, random_state=42)
window_results = {
    "XGBoost, 4 velocity features": evaluate(
        w_test["label"], y_score=score_xgboost(xgb_window, w_test[FEATURE_COLUMNS].values)
    ),
    "transaction_count alone (no model)": {
        "pr_auc": average_precision_score(w_test["label"], w_test["transaction_count"]),
        "roc_auc": roc_auc_score(w_test["label"], w_test["transaction_count"]),
    },
}
pd.DataFrame(window_results).T.round(4)

train: 48,929 windows, 1.54% positive
test:  20,710 windows, 1.05% positive


,pr_auc,roc_auc
"XGBoost, 4 velocity features",0.2507,0.8871
transaction_count alone (no model),0.3777,0.9363


Ranking windows by transaction count alone, with no model, beats XGBoost on all four
features. A window with more transactions has more chances to contain a fraud, so the
"any fraud" label largely encodes window size. A model trained on it learns to flag busy
accounts, not fraud.

## Transaction-level framing: does velocity add anything per transaction?

The alternative is to keep the label on the transaction itself and attach velocity
features as context. `scoring_windows` gives each transaction the features of the
earliest-closing window that contains it, which is what a streaming scorer would have
the first time it could score that transaction.

The split is Phase 3a's: train on `Time` up to its 70th percentile, test on the rest, so
the raw-feature row reproduces Phase 3a's XGBoost exactly.

In [6]:
velocity = scoring_windows(windows)[FEATURE_COLUMNS].add_prefix("w_")
per_txn = transactions.set_index("transaction_id").join(velocity).reset_index()

time_cutoff = per_txn["Time"].quantile(0.70)
t_train = per_txn[per_txn["Time"] <= time_cutoff]
t_test = per_txn[per_txn["Time"] > time_cutoff]

raw_cols = [f"V{i}" for i in range(1, 29)] + ["Amount"]
velocity_cols = list(velocity.columns)
feature_sets = {
    "raw V1-V28 + Amount (Phase 3a)": raw_cols,
    "raw + velocity": raw_cols + velocity_cols,
    "Amount only": ["Amount"],
    "velocity only": velocity_cols,
}

txn_results = {}
for name, cols in feature_sets.items():
    model = fit_xgboost(t_train[cols].values, t_train["Class"].values, random_state=42)
    txn_results[name] = evaluate(t_test["Class"], y_score=score_xgboost(model, t_test[cols].values))

print(f"test: {len(t_test):,} transactions, {int(t_test['Class'].sum())} fraud "
      f"(PR-AUC of a random ranking = {t_test['Class'].mean():.4f})")
pd.DataFrame(txn_results).T[["pr_auc", "roc_auc"]].round(4)

test: 85,439 transactions, 108 fraud (PR-AUC of a random ranking = 0.0013)


,pr_auc,roc_auc
raw V1-V28 + Amount (Phase 3a),0.7988,0.9825
raw + velocity,0.7922,0.9725
Amount only,0.0044,0.6578
velocity only,0.0014,0.5070


Velocity features on their own score at chance: PR-AUC 0.0014 against a random
baseline of 0.0013, and ROC-AUC 0.51. Adding them to the raw features slightly lowers
PR-AUC (0.799 to 0.792).

This is expected from how the data is built. The producer assigns each transaction to a
synthetic account at random (`assign_synthetic_accounts`), independent of whether it is
fraud. There are no compromised accounts, bursts, or account-level patterns for velocity
features to find. The fraud rate is roughly flat across account activity levels:

In [7]:
accounts = per_txn.groupby("account_id").agg(
    transactions=("Class", "size"), fraud_rate=("Class", "mean")
)
accounts["activity quintile"] = pd.qcut(
    accounts["transactions"].rank(method="first"), 5, labels=["lowest", "2", "3", "4", "highest"]
)
by_quintile = per_txn.merge(accounts["activity quintile"], on="account_id").groupby(
    "activity quintile", observed=True
)["Class"].agg(["size", "mean"])
by_quintile.columns = ["transactions", "fraud rate"]
by_quintile["fraud rate"] = by_quintile["fraud rate"].map("{:.3%}".format)
by_quintile

,transactions,fraud rate
activity quintile,,
lowest,2631,0.266%
2,4331,0.092%
3,6672,0.210%
4,12553,0.191%
highest,258620,0.171%


## Key Takeaways

- **The first Phase 3b dataset was invalid.** Live-captured windows held about 0.6% of
  the transactions their labels counted. The batch builder now produces complete windows
  (verified: every transaction in exactly 3), and labels join by transaction ID.
- **The real "any fraud" positive rate is 1.4%**, about 8× the raw rate rather than 116×.
  Even so, the rule mostly labels busy windows: transaction count alone outranks a model
  trained on all four features.
- **Stricter window rules leave too few positives**: 132 windows at a 50% fraud
  threshold, 67 at 100%.
- **Decision: label transactions, not windows.** Phase 3b keeps the transaction's own
  label and treats window features as optional context. This also matches what Phase 4
  has to do: score individual transactions.
- **Velocity features carry no fraud signal in this dataset**, because synthetic accounts
  are assigned at random. That's a property of the simulation, not evidence that velocity
  features are useless. Testing them properly would need a producer that simulates
  account-level fraud behavior.
- **No model is saved from this notebook.** The raw-feature XGBoost from Phase 3a remains
  the model to serve. The earlier `models/xgboost_windowed/2026-09-23/` artifact was
  trained on the invalid capture and should not be used.